# PW05 · Release And Signoff

用途：在 Colab 会话中执行 PW05 release package 与 signoff。PW05 只消费已完成的 PW04 canonical paper exports，以及 PW00/PW02 冻结出的 family manifest、config snapshot 与 thresholds。

边界：这是纯 release / signoff / package 阶段，不重新执行 embed、detect、calibrate、evaluate，也不改写 PW00 到 PW04 的既有工件。

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "PW05_Release_And_Signoff"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow"
REPO_ROOT = Path("/content/ceg_wm_workspace")
SCRIPT_PATH = REPO_ROOT / "paper_workflow" / "scripts" / "PW05_Release_And_Signoff.py"

FAMILY_ID = "paper_eval_family_demo"
FORCE_RERUN = False

def run_checked(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
    DRIVE_MOUNT_STATUS = "mounted"
except Exception as exc:
    DRIVE_MOUNT_STATUS = f"skipped: {type(exc).__name__}: {exc}"

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    origin_url = origin_result.stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

run_checked([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"], cwd=REPO_ROOT)
if (REPO_ROOT / "pyproject.toml").exists():
    run_checked([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], cwd=REPO_ROOT)
else:
    run_checked([sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")], cwd=REPO_ROOT)

print_json(
    "repo_bootstrap",
    {
        "drive_mount_status": DRIVE_MOUNT_STATUS,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "script_path": str(SCRIPT_PATH),
    },
)

In [ ]:
import paper_workflow.scripts.pw05_release_signoff as pw05_module

FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
FAMILY_MANIFEST_PATH = FAMILY_ROOT / "manifests" / "paper_eval_family_manifest.json"
CONFIG_SNAPSHOT_PATH = FAMILY_ROOT / "snapshots" / "config_snapshot.yaml"
PW04_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw04_summary.json"
PW05_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw05_summary.json"
PW02_FINALIZE_MANIFEST_PATH = FAMILY_ROOT / "exports" / "pw02" / "paper_source_finalize_manifest.json"
CONTENT_THRESHOLD_EXPORT_PATH = FAMILY_ROOT / "exports" / "pw02" / "thresholds" / "content" / "thresholds.json"
ATTESTATION_THRESHOLD_EXPORT_PATH = FAMILY_ROOT / "exports" / "pw02" / "thresholds" / "attestation" / "thresholds.json"

PRECHECK_RESULTS = []
QUALITY_RUNTIME_PREFLIGHT = pw05_module._build_quality_runtime_preflight()

def record_precheck(name, ok, detail, diagnostic_only=False):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
        "diagnostic_only": bool(diagnostic_only),
    })

record_precheck("Drive 项目根目录存在", DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
record_precheck("PW05 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck("FAMILY_MANIFEST_PATH 存在", FAMILY_MANIFEST_PATH.exists(), str(FAMILY_MANIFEST_PATH))
record_precheck("CONFIG_SNAPSHOT_PATH 存在", CONFIG_SNAPSHOT_PATH.exists(), str(CONFIG_SNAPSHOT_PATH))
record_precheck("PW04_SUMMARY_PATH 存在", PW04_SUMMARY_PATH.exists(), str(PW04_SUMMARY_PATH))
record_precheck("PW02_FINALIZE_MANIFEST_PATH 存在", PW02_FINALIZE_MANIFEST_PATH.exists(), str(PW02_FINALIZE_MANIFEST_PATH))
record_precheck("content thresholds export 存在", CONTENT_THRESHOLD_EXPORT_PATH.exists(), str(CONTENT_THRESHOLD_EXPORT_PATH))
record_precheck("attestation thresholds export 存在", ATTESTATION_THRESHOLD_EXPORT_PATH.exists(), str(ATTESTATION_THRESHOLD_EXPORT_PATH))
record_precheck(
    "LPIPS 依赖可导入",
    QUALITY_RUNTIME_PREFLIGHT.get("lpips_dependency_ready") is True,
    json.dumps(QUALITY_RUNTIME_PREFLIGHT, ensure_ascii=False, indent=2, sort_keys=True),
    diagnostic_only=True,
 )
record_precheck(
    "CLIP 依赖可导入",
    QUALITY_RUNTIME_PREFLIGHT.get("clip_dependency_ready") is True,
    json.dumps(QUALITY_RUNTIME_PREFLIGHT, ensure_ascii=False, indent=2, sort_keys=True),
    diagnostic_only=True,
 )

PW04_SUMMARY = {}
CANONICAL_METRICS_PATHS = {}
PAPER_TABLE_PATHS = {}
PAPER_FIGURE_PATHS = {}
TAIL_ESTIMATION_PATHS = {}
if PW04_SUMMARY_PATH.exists():
    PW04_SUMMARY = json.loads(PW04_SUMMARY_PATH.read_text(encoding="utf-8"))
    CANONICAL_METRICS_PATHS = dict(PW04_SUMMARY.get("canonical_metrics_paths", {}))
    PAPER_TABLE_PATHS = dict(PW04_SUMMARY.get("paper_tables_paths", {}))
    PAPER_FIGURE_PATHS = dict(PW04_SUMMARY.get("paper_figures_paths", {}))
    TAIL_ESTIMATION_PATHS = dict(PW04_SUMMARY.get("tail_estimation_paths", {}))
    record_precheck("PW04 status == completed", PW04_SUMMARY.get("status") == "completed", json.dumps({"status": PW04_SUMMARY.get("status")}, ensure_ascii=False, indent=2))
    record_precheck("PW04 paper exports completed", PW04_SUMMARY.get("paper_exports_completed") is True, json.dumps({"paper_exports_completed": PW04_SUMMARY.get("paper_exports_completed")}, ensure_ascii=False, indent=2))
    record_precheck("canonical metrics 输出存在", bool(CANONICAL_METRICS_PATHS) and all(Path(str(path)).exists() for path in CANONICAL_METRICS_PATHS.values()), json.dumps(CANONICAL_METRICS_PATHS, ensure_ascii=False, indent=2))
    record_precheck("paper tables 输出存在", bool(PAPER_TABLE_PATHS) and all(Path(str(path)).exists() for path in PAPER_TABLE_PATHS.values()), json.dumps(PAPER_TABLE_PATHS, ensure_ascii=False, indent=2))
    record_precheck("paper figures 输出存在", bool(PAPER_FIGURE_PATHS) and all(Path(str(path)).exists() for path in PAPER_FIGURE_PATHS.values()), json.dumps(PAPER_FIGURE_PATHS, ensure_ascii=False, indent=2))
    record_precheck("tail estimation 输出存在", bool(TAIL_ESTIMATION_PATHS) and all(Path(str(path)).exists() for path in TAIL_ESTIMATION_PATHS.values()), json.dumps(TAIL_ESTIMATION_PATHS, ensure_ascii=False, indent=2))

diagnostic_failures = [item for item in PRECHECK_RESULTS if not item["ok"] and item.get("diagnostic_only") is True]
hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"] and item.get("diagnostic_only") is not True]
print_json(
    "pw05_precheck",
    {
        "items": PRECHECK_RESULTS,
        "hard_failures": hard_fail,
        "diagnostic_failures": diagnostic_failures,
        "quality_runtime_preflight": QUALITY_RUNTIME_PREFLIGHT,
    },
)
if hard_fail:
    raise RuntimeError(f"PW05 precheck failed: {hard_fail}")

In [ ]:
from scripts.notebook_runtime_common import build_repo_import_subprocess_env

COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--family-id",
    FAMILY_ID,
]
if FORCE_RERUN:
    COMMAND.append("--force-rerun")

PW05_RESULT = subprocess.run(
    COMMAND,
    cwd=REPO_ROOT,
    env=build_repo_import_subprocess_env(repo_root=REPO_ROOT),
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

if PW05_RESULT.returncode != 0:
    raise RuntimeError(
        "PW05 failed: "
        f"return_code={PW05_RESULT.returncode} stdout={PW05_RESULT.stdout} stderr={PW05_RESULT.stderr}"
    )
if not PW05_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"PW05 summary file missing after successful execution: {PW05_SUMMARY_PATH}")

PW05_SUMMARY = json.loads(PW05_SUMMARY_PATH.read_text(encoding="utf-8"))
print_json("pw05_summary", PW05_SUMMARY)

In [ ]:
SIGNOFF_REPORT_PATH = Path(str(PW05_SUMMARY["signoff_report_path"]))
RELEASE_MANIFEST_PATH = Path(str(PW05_SUMMARY["release_manifest_path"]))
WORKFLOW_SUMMARY_PATH = Path(str(PW05_SUMMARY["workflow_summary_path"]))
STAGE_MANIFEST_PATH = Path(str(PW05_SUMMARY["stage_manifest_path"]))
PACKAGE_MANIFEST_PATH = Path(str(PW05_SUMMARY["package_manifest_path"]))
PACKAGE_PATH = Path(str(PW05_SUMMARY["package_path"]))

PW05_RESULT_SUMMARY = {
    "summary": json.loads(PW05_SUMMARY_PATH.read_text(encoding="utf-8")),
    "signoff_report": json.loads(SIGNOFF_REPORT_PATH.read_text(encoding="utf-8")),
    "release_manifest": json.loads(RELEASE_MANIFEST_PATH.read_text(encoding="utf-8")),
    "workflow_summary": json.loads(WORKFLOW_SUMMARY_PATH.read_text(encoding="utf-8")),
    "stage_manifest": json.loads(STAGE_MANIFEST_PATH.read_text(encoding="utf-8")),
    "package_manifest": json.loads(PACKAGE_MANIFEST_PATH.read_text(encoding="utf-8")),
    "package_path": str(PACKAGE_PATH),
    "package_exists": PACKAGE_PATH.exists(),
}
print_json("pw05_result_summary", PW05_RESULT_SUMMARY)